# Fairlearn

This notebook walks through deckard's Fairlearn integration for both sklearn and PyTorch workflows, and includes attribute-inference attacks to illustrate privacy risk.

## Navigation

1. Environment and imports
2. Fairlearn-native dataset loading (Adult)
3. FairlearnDataConfig with preprocessing defense
4. sklearn fairness defense (`ExponentiatedGradient`)
5. Attribute-inference attack on sensitive feature
6. PyTorch fairness defense (`AdversarialFairnessClassifier`)
7. Yellowbrick plotter (sklearn fairness experiment)
8. Seaborn summary plots

The dataset source in this notebook is `fairlearn.datasets`, similar to how lifelines-native datasets are used in the survival notebook.

## 1. Environment and Imports

Install optional dependencies before running:

```bash
pip install -e .[fairlearn,seaborn,yellowbrick,torch]
```

In [1]:
# !pip install -e .[fairlearn,seaborn,yellowbrick,torch]

In [2]:
# All imports moved to the top for clarity and reproducibility
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score
from deckard.data import FairlearnDataConfig
from deckard.model import FairlearnModelConfig, FairlearnPytorchModelConfig
from deckard.model.pytorch import TinyNet
from deckard.experiment import TorchExperimentConfig
from deckard.plot.seaborn_plots import SeabornPlotConfig, SeabornPlotConfigList
from deckard.plot.yellowbrick_plots import YellowbrickPlotConfig

NOTEBOOK_ROOT = Path(".")
ARTIFACT_DIR = NOTEBOOK_ROOT / "build" / "fairlearn"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

# Canonical output file paths for each step
data_file = ARTIFACT_DIR / "fairlearn_data.pkl"
fair_data_file = ARTIFACT_DIR / "fairlearn_fair_data.pkl"
sklearn_model_file = ARTIFACT_DIR / "fairlearn_sklearn_model.pkl"
torch_model_file = ARTIFACT_DIR / "fairlearn_torch_model.pt"
sklearn_score_file = ARTIFACT_DIR / "fairlearn_sklearn_scores.csv"
torch_score_file = ARTIFACT_DIR / "fairlearn_torch_scores.csv"
attack_file = ARTIFACT_DIR / "fairlearn_attack_results.pkl"
plot_file = ARTIFACT_DIR / "fairlearn_attack_plot.png"

print(f"Artifacts will be written under: {ARTIFACT_DIR}")

/Users/c.meyers/.pyenv/versions/deckard/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Artifacts will be written under: build/fairlearn


## 2. Adult Dataset Loading

This section uses deckard's built-in `adult` loader, which now performs the Adult target encoding, sensitive-feature encoding, and categorical expansion centrally in `DataConfig`.

In [3]:
adult_data = FairlearnDataConfig(
    dataset_name="adult",
    train_size=0.8,
    test_size=0.2,
    random_state=42,
    classifier=True,
    sensitive_columns="sex",
)
data_scores = adult_data(data_file=data_file.as_posix())
print(data_scores)

[ERROR] Non-flat or nested value in FairlearnDataConfig score_dict for key 'fairness_scores': {'training_num_classes': 2, 'training_class_count_min': 9349, 'training_class_count_max': 29724, 'training_class_imbalance_ratio': 3.1793774735265803, 'training_mutual_information_mean': 0.008113330293080452, 'training_mutual_information_max': 0.10884857622098965} (type: <class 'dict'>)


  File "/Users/c.meyers/.pyenv/versions/3.10.20/lib/python3.10/runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/Users/c.meyers/.pyenv/versions/3.10.20/lib/python3.10/runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "/Users/c.meyers/.pyenv/versions/deckard/lib/python3.10/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/Users/c.meyers/.pyenv/versions/deckard/lib/python3.10/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/Users/c.meyers/.pyenv/versions/deckard/lib/python3.10/site-packages/ipykernel/kernelapp.py", line 758, in start
    self.io_loop.start()
  File "/Users/c.meyers/.pyenv/versions/deckard/lib/python3.10/site-packages/tornado/platform/asyncio.py", line 211, in start
    self.asyncio_loop.run_forever()
  File "/Users/c.meyers/.pyenv/versions/3.10.20/lib/python3.10/asyncio/base_events.py", line 603, in run_

ValueError: FairlearnDataConfig score_dict key 'fairness_scores' contains a non-flat or nested value: {'training_num_classes': 2, 'training_class_count_min': 9349, 'training_class_count_max': 29724, 'training_class_imbalance_ratio': 3.1793774735265803, 'training_mutual_information_mean': 0.008113330293080452, 'training_mutual_information_max': 0.10884857622098965}

## 3. FairlearnDataConfig with Preprocessing Defense

`FairlearnDataConfig` can inject a Fairlearn preprocessing defense (`CorrelationRemover`) while preserving sensitive-feature caches for fairness and attack scoring.

In [ ]:
from deckard.data import FairlearnDataConfig

fair_data = FairlearnDataConfig(
    dataset_name="adult",
    classifier=True,
    stratify=False,
    train_size=0.8,
    test_size=0.2,
    random_state=42,
    sensitive_columns=["sex"],
    fairness_defense={
        "step_name": "fairness_correlation_remover",
        "name": "fairlearn.preprocessing.CorrelationRemover",
        "alpha": 1.0,
    },
)

fair_data_scores = fair_data(data_file=fair_data_file.as_posix(), score_file=sklearn_score_file.as_posix())

print("Train shape:", fair_data.X_train.shape)
print("Test shape:", fair_data.X_test.shape)
print("Sensitive train sample:", fair_data._sensitive_train.iloc[:5].tolist())
print(fair_data_scores.keys())

Train shape: (39073, 103)
Test shape: (9769, 103)
Sensitive train sample: ['0', '0', '0', '1', '0']
dict_keys(['data_load_time', 'data_sample_time', 'pipeline_fit_time', 'pipeline_fit_n', 'pipeline_transform_time', 'pipeline_transform_n', 'fairness_scores'])


## 4. sklearn Fairness Workflow on Defended Data

This section trains a sklearn model on the Fairlearn-defended Adult dataset from Section 3 and reports both utility and group-level accuracy gap metrics.

In [ ]:


sk_fair_model = FairlearnModelConfig(
    model_type="sklearn.linear_model.LogisticRegression",
    classifier=True,
    model_params={"max_iter": 200},
    data=fair_data,
)

sk_fair_scores = sk_fair_model(fair_data, model_file=sklearn_model_file.as_posix(), score_file=sklearn_score_file.as_posix(), )

if "accuracy" not in 

{'0': {0: '0.06903499999999951', 1: '0.0007450000000002177', 2: '0.03589600000000104', 3: '39073', 4: '0.006774000000000058', 5: '9769', 6: "{'training_num_classes': 2, 'training_class_count_min': 9397, 'training_class_count_max': 29676, 'training_class_imbalance_ratio': 3.158029158241992, 'training_mutual_information_mean': 0.03075386955040607, 'training_mutual_information_max': 0.10708002519948989}"}, 'accuracy': 0.8127, 'precision': 0.7975, 'recall': 0.8127, 'f1': 0.7979, 'roc_auc': 0.7821, 'log_loss': 0.4387, 'training_time': 0.28818800000000167, 'training_n': 39073, 'prediction_time': 0.0024160000000001958, 'prediction_n': 9769, 'prediction_score_time': 0.009133999999999531}


## 5. PyTorch Fairness Workflow

This section reuses the Fairlearn-preprocessed Adult dataset from Section 3 and fits a PyTorch model with fairness-aware group scoring. The Fairlearn defense here is the same `CorrelationRemover` preprocessing step, now evaluated through a torch model workflow.

In [ ]:

from deckard.data.fairness_pytorch import TinyFairness, FairlearnPytorchDataConfig

# Use the same data config as sklearn, but for torch
torch_fair_data = FairlearnPytorchDataConfig(dataset_name="deckard.data.fairness_pytorch.TinyFairness", sensitive_columns=["_sensitive"])
torch_fair_data(data_file=(ARTIFACT_DIR / "torch_fair_data.pkl").as_posix())


torch_fair_model = FairlearnPytorchModelConfig(
    model_type="torch.nn.Linear",
    model_params={"in_features": int(torch_fair_data.X_train.shape[1]), "out_features": 2},
    classifier=True,
    criterion="CrossEntropyLoss",
    optimizer={"name": "torch.optim.Adam", "lr": 1e-3},
    fit_params={"nb_epochs": 2, "batch_size": 128},
    data=torch_fair_data,
    device="cpu",
)

torch_fair_scores = torch_fair_model(torch_fair_data, model_file=torch_model_file.as_posix(), score_file=torch_score_file.as_posix())

interesting_torch = [
    k for k in sorted(torch_fair_scores.keys())
    if "accuracy" in k or "sensitive_feature" in k or "f1" in k
]
print("torch fairness score keys:", interesting_torch[:20])
print({k: torch_fair_scores[k] for k in interesting_torch[:8] if isinstance(torch_fair_scores[k], (int, float))})

InstantiationException: Error locating target '', set env var HYDRA_FULL_ERROR=1 to see chained exception.

## 6. Torch Fairness Experiment: Before/After Defense Comparison and Plots

This section uses `TorchExperimentConfig` to run before/after defense experiments with TinyNet and TinyFairness, then visualizes group accuracy improvements using seaborn and ROC AUC using Yellowbrick for both sklearn and torch models.

In [ ]:
# Create synthetic fairness dataset using Deckard objects only
dataset = TinyFairness(num_samples=200, n_features=8, random_state=42)
X = dataset._X
y = dataset._y
sensitive = dataset._sensitive

# Split into train/test
X_train, X_test, y_train, y_test, s_train, s_test = train_test_split(
    X, y, sensitive, test_size=0.2, random_state=42, stratify=y
)

# Prepare data config for TorchExperimentConfig (before defense)
data_cfg = dict(
    dataset_name="deckard.model.pytorch.TinyFairness",
    sensitive_columns=[0],  # index of sensitive column in TinyFairness
    classifier=True,
    train_size=0.8,
    test_size=0.2,
    random_state=42,
    stratify=True,
    # No fairness_defense for baseline
    )
data_before = FairlearnDataConfig(**data_cfg)
data_before = data_before(data_file=(ARTIFACT_DIR / "fairlearn_torch_data_before.pkl").as_posix())

# Model config (TinyNet, no defense)
model_cfg = {
    "model_type": TinyNet,
    "classifier": True,
    "model_params": {"input_dim": int(X_train.shape[1]), "hidden_dim": 16, "output_dim": 2},
    "fit_params": {"nb_epochs": 2, "batch_size": 32},
    "criterion": "CrossEntropyLoss",
    "optimizer": {"name": "torch.optim.Adam", "lr": 1e-3},
    "device": "cpu",
}
model_before = FairlearnPytorchModelConfig(**model_cfg)

# Run baseline experiment (no defense)
exp_before = TorchExperimentConfig(data=data_before, model=model_before)
scores_before = exp_before()

# Prepare data config for TorchExperimentConfig (with defense)
data_cfg_def = dict(data_cfg)
data_cfg_def["fairness_defense"] = {
    "step_name": "fairness_correlation_remover",
    "name": "fairlearn.preprocessing.CorrelationRemover",
    "alpha": 1.0,
}
data_after = FairlearnDataConfig(**data_cfg_def)
data_after = data_after(data_file=(ARTIFACT_DIR / "fairlearn_torch_data_after.pkl").as_posix())

# Model config (TinyNet, with defense)
model_after = FairlearnPytorchModelConfig(**model_cfg)

# Run defended experiment
exp_after = TorchExperimentConfig(data=data_after, model=model_after)
scores_after = exp_after()

print("Torch baseline (no defense) scores:", scores_before)
print("Torch with defense scores:", scores_after)

# Seaborn comparison plot (accuracy/group accuracy)
def extract_group_accuracies(scores, prefix="group_"):
    return {k: v for k, v in scores.items() if k.startswith(prefix)}

group_acc_before = extract_group_accuracies(scores_before)
group_acc_after = extract_group_accuracies(scores_after)

df_plot = pd.DataFrame({
    "Group": list(group_acc_before.keys()) + list(group_acc_after.keys()),
    "Accuracy": list(group_acc_before.values()) + list(group_acc_after.values()),
    "Defense": ["Before"] * len(group_acc_before) + ["After"] * len(group_acc_after),
})
plt.figure(figsize=(6,4))
sns.barplot(data=df_plot, x="Group", y="Accuracy", hue="Defense")
plt.title("Group Accuracy Before/After Defense (Torch)")
plt.show()

# Yellowbrick ROC AUC for sklearn model
plot_config_sklearn = YellowbrickPlotConfig(
    experiment=None,
    model_type='sklearn.linear_model.LogisticRegression',
    X_train=fair_data.X_train, y_train=fair_data.y_train,
    X_test=fair_data.X_test, y_test=fair_data.y_test,
    outpath=(ARTIFACT_DIR / 'sklearn_roc_auc.png').as_posix(),
)
plot_config_sklearn()

# Yellowbrick ROC AUC for torch model (after defense)
plot_config_torch = YellowbrickPlotConfig(
    experiment=exp_after,
    model_type=TinyNet,
    X_train=data_after.X_train, y_train=data_after.y_train,
    X_test=data_after.X_test, y_test=data_after.y_test,
    outpath=(ARTIFACT_DIR / 'torch_roc_auc.png').as_posix(),
)
plot_config_torch()

## 7. Seaborn and Yellowbrick Summary Plots

This section summarizes fairness and privacy-risk metrics from the sklearn and torch sections, including before/after defense group accuracy and ROC AUC plots.